# Alexnet by PyTorch

Source : https://pytorch.org/hub/pytorch_vision_alexnet/

modules

In [1]:
import os
from typing import Tuple, Callable
from PIL import Image
import torch
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

from datasets import DATASET_1, DATASET_2, CustomImageDataset, get_label_data_from_filename

device

In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"{device} device available")

mps device available


modèle

In [3]:
model_alexnet = torch.hub.load('pytorch/vision:v0.10.0', 'alexnet', pretrained=True)
model_alexnet.eval()
model_alexnet.to(device)

Using cache found in /Users/me/.cache/torch/hub/pytorch_vision_v0.10.0
/Users/me/opt/anaconda3/envs/cnamrcp209/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/me/opt/anaconda3/envs/cnamrcp209/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

Chargement du dataset

In [4]:
class CustomImageDataset(Dataset):
    @classmethod
    def use_decode_image(cls) -> bool:
        return True


    def __init__(
            self,
            img_dir: str,
            transform: T.Compose=None,
            extension: str=".jpg",
            to_rgb: bool = True,
            dataset_mode: bool = True,
            only_label_idx: bool = True,
            get_label_data: Callable = None,
            max_length_padding_filename: int = 40,
            ) -> None:
        #self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.extension = extension.lower()
        self.files = [f for f in os.listdir(img_dir) if f.lower().endswith(self.extension)]
        self.files = sorted(self.files)
        self.to_rgb = to_rgb
        self.dataset_mode = dataset_mode
        self.only_label_idx = only_label_idx
        self.get_label_data = get_label_data
        self.max_length_padding_filename = max_length_padding_filename


    def __len__(self) -> int:
        return len(self.files)


    def __getitem__(self, idx: int) \
        -> Tuple[torch.Tensor, int]|Tuple[torch.Tensor, int, str, int]|Tuple[torch.Tensor, torch.Tensor, int, str, int]:
        if idx >= len(self):
            return None, None, None
        
        image, image_trfm = self.get_image(self.files[idx], to_rgb=self.to_rgb)
        if self.get_label_data:
            data = self.get_label_data(self.files[idx])
            label_idx = data[1]
            label_code = data[0]
        else:
            label_idx = None
            label_code = None
        
        if self.dataset_mode:
            if self.only_label_idx:
                return image_trfm, label_idx
            else:
                return image_trfm, label_idx, label_code, idx
        else:
            return image, image_trfm, label_idx, label_code, idx
    

    def get_image(self, name: str, to_rgb: bool = True) -> Tuple[torch.Tensor, torch.Tensor]:
        assert name in self.files, f"{name} unknown"

        img_path = os.path.join(self.img_dir, name)
        image = Image.open(img_path)

        # If the image is not a RGB but a 1 channel grey
        if to_rgb and len(image.getbands()) == 1:
            image_mono = image.convert("L")
            image = Image.merge("RGB", (image_mono, image_mono, image_mono))
            
        image_trfm = self.transform(image) if self.transform else None

        return image, image_trfm

In [5]:
transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

DATASET = DATASET_1
get_label_data = lambda f: get_label_data_from_filename(f, DATASET["path"])
dataset_path = DATASET["mounted_path"] if os.path.exists(DATASET["mounted_path"]) else DATASET["path"]
print("Using Dataset (name, path)", DATASET["name"], dataset_path)

dataset = CustomImageDataset(
    dataset_path,
    transform=transforms,
    extension="JPEG",
    dataset_mode=True,
    only_label_idx=True,
    get_label_data=get_label_data,
    )

Using Dataset (name, path) imagenet_val_images (50K images) /Users/me/Documents/Work/Dev/_data/imagenet_val_images


In [6]:
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [7]:
all_predictions_batch = []
all_expecteds_batch = []
for i, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
    output = model_alexnet.forward(batch[0].to(device)) # To carefully transfer to CPU before append in list
    output = output.to("cpu").detach()
    probabilities = torch.nn.functional.softmax(output, dim=0)
    top5_prob, top5_catid = torch.topk(probabilities, 5)
    all_predictions_batch.append((top5_prob.tolist(), top5_catid.tolist()))
    all_expecteds_batch.append(batch[1].tolist())

  0%|          | 0/1563 [00:00<?, ?it/s]

In [26]:
from collections import Counter
distribution_computed_top1 = Counter()
distribution_computed_top5 = Counter()
distribution_expected = Counter()

right_top1 = 0
right_top5 = 0

for (predictions_batch_prob, predictions_batch_idx), expected_batch in zip(all_predictions_batch, all_expecteds_batch):
    predictions_batch_idx = torch.tensor(predictions_batch_idx)
    expected_batch = torch.tensor(expected_batch).reshape(-1, 1)
    expected_in_top1 = (predictions_batch_idx[:, 1] - expected_batch) == 0
    expected_in_top5 = (predictions_batch_idx - expected_batch) == 0

    right_top1 += expected_in_top1.sum().item()
    right_top5 += expected_in_top5.sum().item()

    distribution_computed_top1.update(predictions_batch_idx[:, 0].ravel().tolist())
    distribution_computed_top5.update(predictions_batch_idx.ravel().tolist())
    distribution_expected.update(expected_batch.ravel().tolist())


print(f"Justesse top1: {right_top1} ({right_top1 / len(dataset) * 100:.2f}%)")
print(f"Justesse top5: {right_top5} ({right_top5 / len(dataset) * 100:.2f}%)")

Justesse top1: 5917 (11.83%)
Justesse top5: 32927 (65.85%)


In [27]:
print(distribution_computed_top1.total())
print(distribution_computed_top5.total())

50000
250000


In [29]:
distribution_computed_top1.most_common(10)

[(500, 158),
 (548, 139),
 (640, 131),
 (669, 128),
 (685, 128),
 (974, 120),
 (607, 120),
 (926, 119),
 (535, 114),
 (153, 113)]

In [30]:
distribution_computed_top5.most_common(10)

[(685, 489),
 (500, 487),
 (548, 486),
 (669, 481),
 (393, 460),
 (564, 450),
 (416, 445),
 (979, 443),
 (689, 442),
 (149, 440)]

In [36]:
list(distribution_expected.items())[:10]

[(240, 50),
 (616, 50),
 (117, 50),
 (926, 50),
 (170, 50),
 (987, 50),
 (998, 50),
 (339, 50),
 (171, 50),
 (338, 50)]